# 04 — Data Storage: Parquet (Bab 2.3)Notebook ini mengimplementasikan penyimpanan data ke format **Apache Parquet** dengan **partisi berdasarkan Product_Category**.

## 4.1 Inisialisasi & Load Data

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import col, to_date

spark = SparkSession.builder \
    .appName("04_DataStorage") \
    .master("local[*]") \
    .config("spark.driver.memory", "1g") \
    .getOrCreate()

schema = StructType([
    StructField("Transaction_ID", IntegerType(), False),
    StructField("Date", StringType(), False),
    StructField("Customer_ID", StringType(), False),
    StructField("Gender", StringType(), False),
    StructField("Age", IntegerType(), False),
    StructField("Product_Category", StringType(), False),
    StructField("Quantity", IntegerType(), False),
    StructField("Price_per_Unit", IntegerType(), False),
    StructField("Total_Amount", IntegerType(), False)
])

df = spark.read.csv("/data/retail_sales_dataset.csv", header=True, schema=schema)
df = df.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))
print(f"✅ Loaded {df.count()} rows")

✅ Loaded 1000 rows


## 4.2 Simpan ke Parquet (Partitioned)

In [2]:
PARQUET_PATH = "/output/retail_parquet"

# Tulis ke Parquet, partisi berdasarkan Product_Category
df.write \
    .mode("overwrite") \
    .partitionBy("Product_Category") \
    .parquet(PARQUET_PATH)

print(f"✅ Data disimpan ke: {PARQUET_PATH}")
print(f"   Partisi: Product_Category (Beauty, Clothing, Electronics)")

✅ Data disimpan ke: /output/retail_parquet
   Partisi: Product_Category (Beauty, Clothing, Electronics)


## 4.3 Verifikasi: Baca Kembali dari Parquet

In [3]:
# Baca kembali
df_parquet = spark.read.parquet(PARQUET_PATH)

print(f"✅ Parquet dibaca kembali: {df_parquet.count()} rows")
print(f"   Kolom: {df_parquet.columns}")
df_parquet.printSchema()
df_parquet.show(5)

✅ Parquet dibaca kembali: 1000 rows
   Kolom: ['Transaction_ID', 'Date', 'Customer_ID', 'Gender', 'Age', 'Quantity', 'Price_per_Unit', 'Total_Amount', 'Product_Category']
root
 |-- Transaction_ID: integer (nullable = true)
 |-- Date: date (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Price_per_Unit: integer (nullable = true)
 |-- Total_Amount: integer (nullable = true)
 |-- Product_Category: string (nullable = true)

+--------------+----------+-----------+------+---+--------+--------------+------------+----------------+
|Transaction_ID|      Date|Customer_ID|Gender|Age|Quantity|Price_per_Unit|Total_Amount|Product_Category|
+--------------+----------+-----------+------+---+--------+--------------+------------+----------------+
|             2|2023-02-27|    CUST002|Female| 26|       2|           500|        1000|        Clothing|
|             4|2023-05-

## 4.4 Perbandingan Performa CSV vs Parquet

In [4]:
import time

# Benchmark: CSV read
t0 = time.time()
df_csv = spark.read.csv("/data/retail_sales_dataset.csv", header=True, schema=schema)
df_csv.count()
csv_time = time.time() - t0

# Benchmark: Parquet read
t0 = time.time()
df_pq = spark.read.parquet(PARQUET_PATH)
df_pq.count()
pq_time = time.time() - t0

print("=== PERBANDINGAN PERFORMA ===")
print(f"   CSV read time     : {csv_time:.3f}s")
print(f"   Parquet read time : {pq_time:.3f}s")
print(f"   Speedup           : {csv_time/pq_time:.1f}x lebih cepat" if pq_time > 0 else "   (terlalu cepat untuk dibandingkan)")

=== PERBANDINGAN PERFORMA ===
   CSV read time     : 0.096s
   Parquet read time : 0.120s
   Speedup           : 0.8x lebih cepat


## 4.5 Demo: Partition PruningKeunggulan partisi — hanya baca data yang dibutuhkan.

In [5]:
# Hanya baca kategori 'Electronics' (partition pruning)
t0 = time.time()
df_elec = spark.read.parquet(PARQUET_PATH).filter(col("Product_Category") == "Electronics")
count_elec = df_elec.count()
prune_time = time.time() - t0

print(f"✅ Partition Pruning: hanya baca Electronics")
print(f"   Rows  : {count_elec}")
print(f"   Time  : {prune_time:.3f}s")
print(f"   → Spark hanya membaca folder 'Product_Category=Electronics', skip sisanya")

✅ Partition Pruning: hanya baca Electronics
   Rows  : 342
   Time  : 0.148s
   → Spark hanya membaca folder 'Product_Category=Electronics', skip sisanya
